In [6]:
import librosa
import soundfile as sf
import numpy as np
from scipy import stats
import noisereduce as nr
import warnings

# 过滤可选警告
warnings.filterwarnings("ignore", message="IProgress not found. Please update jupyter and ipywidgets")

def extract_statistical_features_with_details(audio_path, target_sr=16000):
    """
    提取语音统计特征并返回分类别统计数据
    :param audio_path: 音频文件路径（wav格式）
    :param target_sr: 目标采样率（与WavLM一致为16000）
    :return: 
        - final_features: 标准化后的最终统计特征向量（numpy数组）
        - feature_stats: 分类别统计数据字典（含每类特征的详细统计量）
    """
    # -------------------------- 步骤1：音频预处理 --------------------------
    y, orig_sr = librosa.load(audio_path, sr=None)
    y = librosa.resample(y, orig_sr=orig_sr, target_sr=target_sr)
    y = nr.reduce_noise(y=y, sr=target_sr)
    
    # 分帧参数
    frame_length = int(target_sr * 0.025)
    hop_length = int(target_sr * 0.01)
    window = np.hamming(frame_length)

    # -------------------------- 步骤2：提取基础声学特征 --------------------------
    # 1. 时域特征
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_length, hop_length=hop_length)[0]  # 过零率
    energy = np.array([np.sum(np.square(y[i:i+frame_length] * window)) for i in range(0, len(y)-frame_length, hop_length)])  # 短时能量
    speech_rate = len(zcr) / librosa.get_duration(y=y, sr=target_sr) if len(y) > 0 else 0  # 语速（标量）

    # 2. 频域特征
    mfcc = librosa.feature.mfcc(y=y, sr=target_sr, n_mfcc=13, n_fft=frame_length, hop_length=hop_length)  # 13维MFCC
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=target_sr, n_fft=frame_length, hop_length=hop_length)[0]  # 频谱质心

    # 3. 韵律特征（基频F0）
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0 = f0[~np.isnan(f0)]  # 去除无声段的NaN值

    # -------------------------- 步骤3：计算分类别统计量 --------------------------
    def calc_stats(feat, feat_name):
        """计算单个特征的6类统计量，并返回带名称的字典"""
        if len(feat) == 0:
            return {
                f"{feat_name}_mean": 0.0,
                f"{feat_name}_std": 0.0,
                f"{feat_name}_max": 0.0,
                f"{feat_name}_min": 0.0,
                f"{feat_name}_skewness": 0.0,
                f"{feat_name}_kurtosis": 0.0
            }
        return {
            f"{feat_name}_mean": float(np.mean(feat)),
            f"{feat_name}_std": float(np.std(feat)),
            f"{feat_name}_max": float(np.max(feat)),
            f"{feat_name}_min": float(np.min(feat)),
            f"{feat_name}_skewness": float(stats.skew(feat)),
            f"{feat_name}_kurtosis": float(stats.kurtosis(feat))
        }
    
    # 初始化分类别统计字典
    feature_stats = {
        "时域特征": {},
        "频域特征": {},
        "韵律特征": {},
        "全局汇总": {}
    }

    # 1. 时域特征统计
    feature_stats["时域特征"]["过零率(ZCR)"] = calc_stats(zcr, "ZCR")
    feature_stats["时域特征"]["短时能量(Energy)"] = calc_stats(energy, "Energy")
    feature_stats["时域特征"]["语速(SpeechRate)"] = {"SpeechRate_value": float(speech_rate)}  # 标量无统计量

    # 2. 频域特征统计
    feature_stats["频域特征"]["频谱质心(SpectralCentroid)"] = calc_stats(spectral_centroid, "SpectralCentroid")
    mfcc_stats = {}
    for i in range(13):
        mfcc_dim = mfcc[i]
        mfcc_stats[f"MFCC_{i+1}"] = calc_stats(mfcc_dim, f"MFCC_{i+1}")
    feature_stats["频域特征"]["13维MFCC"] = mfcc_stats

    # 3. 韵律特征统计
    feature_stats["韵律特征"]["基频(F0)"] = calc_stats(f0, "F0")

    # -------------------------- 步骤4：拼接并标准化最终特征 --------------------------
    # 整合所有统计量值（用于最终特征向量）
    stat_values = []
    # 时域特征
    stat_values.extend(list(feature_stats["时域特征"]["过零率(ZCR)"].values()))
    stat_values.extend(list(feature_stats["时域特征"]["短时能量(Energy)"].values()))
    stat_values.append(speech_rate)
    # 频域特征
    stat_values.extend(list(feature_stats["频域特征"]["频谱质心(SpectralCentroid)"].values()))
    for i in range(13):
        stat_values.extend(list(feature_stats["频域特征"]["13维MFCC"][f"MFCC_{i+1}"].values()))
    # 韵律特征
    stat_values.extend(list(feature_stats["韵律特征"]["基频(F0)"].values()))

    # 标准化
    stat_values = np.array(stat_values)
    std_val = np.std(stat_values)
    if std_val != 0:
        final_features = (stat_values - np.mean(stat_values)) / std_val
    else:
        final_features = stat_values - np.mean(stat_values)

    # 全局汇总统计
    feature_stats["全局汇总"] = {
        "原始特征总维度": len(stat_values),
        "标准化后均值": float(np.mean(final_features)),
        "标准化后标准差": float(np.std(final_features)),
        "标准化后最大值": float(np.max(final_features)),
        "标准化后最小值": float(np.min(final_features))
    }

    return final_features, feature_stats

# 测试示例：输出每类特征的详细统计数据
if __name__ == "__main__":
    audio_path = "/home/user512-003/songcw/code/scw-ser/data/wav/Ses01F_impro01/Ses01F_impro01_F000.wav"
    final_features, feature_stats = extract_statistical_features_with_details(audio_path)

    # -------------------------- 打印分类别统计数据 --------------------------
    print("="*80)
    print("1. 时域特征统计数据（反映语音节奏/强度）")
    print("-"*50)
    for feat_name, stats_dict in feature_stats["时域特征"].items():
        print(f"\n{feat_name}:")
        for stat_name, value in stats_dict.items():
            print(f"  {stat_name}: {value:.6f}")

    print("\n" + "="*80)
    print("2. 频域特征统计数据（反映语音频谱/音色）")
    print("-"*50)
    # 打印频谱质心
    print("\n频谱质心(SpectralCentroid):")
    for stat_name, value in feature_stats["频域特征"]["频谱质心(SpectralCentroid)"].items():
        print(f"  {stat_name}: {value:.6f}")
    # 打印前3维MFCC（避免输出过长，其余维度可按需查看）
    print("\n13维MFCC（仅展示前3维，共13维）:")
    for i in range(3):
        mfcc_name = f"MFCC_{i+1}"
        print(f"\n  {mfcc_name}:")
        for stat_name, value in feature_stats["频域特征"]["13维MFCC"][mfcc_name].items():
            print(f"    {stat_name}: {value:.6f}")

    print("\n" + "="*80)
    print("3. 韵律特征统计数据（反映语音语调）")
    print("-"*50)
    print("\n基频(F0):")
    for stat_name, value in feature_stats["韵律特征"]["基频(F0)"].items():
        print(f"  {stat_name}: {value:.6f}")

    print("\n" + "="*80)
    print("4. 全局汇总统计")
    print("-"*50)
    for stat_name, value in feature_stats["全局汇总"].items():
        print(f"{stat_name}: {value:.6f}")

    # 可选：打印最终标准化特征向量
    print("\n" + "="*80)
    print(f"5. 最终标准化统计特征向量维度: {final_features.shape[0]}")
    print(f"前10维特征值: {final_features[:10]}")

1. 时域特征统计数据（反映语音节奏/强度）
--------------------------------------------------

过零率(ZCR):
  ZCR_mean: 0.144346
  ZCR_std: 0.157780
  ZCR_max: 0.725000
  ZCR_min: 0.010000
  ZCR_skewness: 2.050004
  ZCR_kurtosis: 3.550289

短时能量(Energy):
  Energy_mean: 0.006110
  Energy_std: 0.017299
  Energy_max: 0.093842
  Energy_min: 0.000002
  Energy_skewness: 3.309787
  Energy_kurtosis: 10.838364

语速(SpeechRate):
  SpeechRate_value: 100.228083

2. 频域特征统计数据（反映语音频谱/音色）
--------------------------------------------------

频谱质心(SpectralCentroid):
  SpectralCentroid_mean: 1505.707012
  SpectralCentroid_std: 1056.362468
  SpectralCentroid_max: 5287.806698
  SpectralCentroid_min: 320.338019
  SpectralCentroid_skewness: 1.761660
  SpectralCentroid_kurtosis: 2.658634

13维MFCC（仅展示前3维，共13维）:

  MFCC_1:
    MFCC_1_mean: -756.238892
    MFCC_1_std: 110.808731
    MFCC_1_max: -536.721130
    MFCC_1_min: -930.668518
    MFCC_1_skewness: 0.464296
    MFCC_1_kurtosis: -0.919171

  MFCC_2:
    MFCC_2_mean: 99.633400
    MF

In [10]:
import librosa
import numpy as np
from scipy import stats
import noisereduce as nr
import pyworld as pw
import warnings

# 过滤可选警告（不影响核心功能）
warnings.filterwarnings("ignore", message="IProgress not found. Please update jupyter and ipywidgets")
warnings.filterwarnings("ignore", message="PySoundFile failed. Trying audioread instead.")

# -------------------------- 全局参数配置（与WavLM特征提取对齐） --------------------------
TARGET_SR = 16000  # 目标采样率
FRAME_LENGTH = int(TARGET_SR * 0.025)  # 25ms帧长
HOP_LENGTH = int(TARGET_SR * 0.01)     # 10ms帧移
WINDOW = np.hamming(FRAME_LENGTH)      # 汉明窗（减少频谱泄漏）

# -------------------------- 通用工具函数：6类统计量计算 --------------------------
def calc_statistics(feat, feat_name):
    """
    计算单个特征序列的6类核心统计量（均值/标准差/最大值/最小值/偏度/峰度）
    兼容空值/NaN值处理，避免计算报错
    """
    # 处理空特征/全NaN特征的边界情况
    if len(feat) == 0 or np.all(np.isnan(feat)):
        stats_dict = {
            f"{feat_name}_mean": 0.0,
            f"{feat_name}_std": 0.0,
            f"{feat_name}_max": 0.0,
            f"{feat_name}_min": 0.0,
            f"{feat_name}_skewness": 0.0,
            f"{feat_name}_kurtosis": 0.0
        }
        return stats_dict, list(stats_dict.values())
    
    # 去除有效序列中的NaN值
    feat = feat[~np.isnan(feat)]
    
    # 计算6类统计量
    stats_dict = {
        f"{feat_name}_mean": float(np.mean(feat)),
        f"{feat_name}_std": float(np.std(feat)),
        f"{feat_name}_max": float(np.max(feat)),
        f"{feat_name}_min": float(np.min(feat)),
        f"{feat_name}_skewness": float(stats.skew(feat)),
        f"{feat_name}_kurtosis": float(stats.kurtosis(feat))
    }
    return stats_dict, list(stats_dict.values())

# -------------------------- 步骤1：音频预处理（核心修复：类型转换） --------------------------
def preprocess_audio(audio_path):
    """
    标准化音频预处理：统一采样率 + 专业去噪 + 类型转换为float64（适配pyworld）
    """
    # 读取音频并统一重采样至目标采样率
    y, orig_sr = librosa.load(audio_path, sr=None)
    y = librosa.resample(y, orig_sr=orig_sr, target_sr=TARGET_SR)
    
    # 核心修复：将音频信号转为float64（double），适配pyworld的类型要求
    y = y.astype(np.float64)
    
    # 基于noisereduce的语音去噪（保留有效发声信息）
    y = nr.reduce_noise(y=y, sr=TARGET_SR)
    
    return y, TARGET_SR

# -------------------------- 特征模块1：时域声学特征 --------------------------
def extract_time_domain_features(y, sr):
    """提取：过零率(ZCR)、短时能量、语速"""
    time_feats_dict = {}
    time_feats_values = []
    
    # 1. 过零率（ZCR）
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]
    zcr_stats, zcr_vals = calc_statistics(zcr, "ZCR")
    time_feats_dict["过零率(ZCR)"] = zcr_stats
    time_feats_values.extend(zcr_vals)
    
    # 2. 短时能量
    short_term_energy = np.array([
        np.sum(np.square(y[i:i+FRAME_LENGTH] * WINDOW)) 
        for i in range(0, len(y)-FRAME_LENGTH, HOP_LENGTH)
    ])
    energy_stats, energy_vals = calc_statistics(short_term_energy, "ShortTermEnergy")
    time_feats_dict["短时能量(ShortTermEnergy)"] = energy_stats
    time_feats_values.extend(energy_vals)
    
    # 3. 语速（帧数量/音频总时长）
    total_duration = librosa.get_duration(y=y, sr=sr)
    speech_rate = len(zcr) / total_duration if (len(y) > 0 and total_duration > 0) else 0.0
    time_feats_dict["语速(SpeechRate)"] = {"SpeechRate_value": float(speech_rate)}
    time_feats_values.append(float(speech_rate))
    
    return time_feats_dict, time_feats_values

# -------------------------- 特征模块2：频域与倒谱特征 --------------------------
def extract_frequency_cepstral_features(y, sr):
    """提取：频谱质心、静态MFCC、Delta MFCC、Delta-Delta MFCC"""
    freq_feats_dict = {}
    freq_feats_values = []
    
    # 1. 频谱质心
    spectral_centroid = librosa.feature.spectral_centroid(
        y=y, sr=sr, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH
    )[0]
    sc_stats, sc_vals = calc_statistics(spectral_centroid, "SpectralCentroid")
    freq_feats_dict["频谱质心(SpectralCentroid)"] = sc_stats
    freq_feats_values.extend(sc_vals)
    
    # 2. 静态MFCC + 动态差分MFCC（13维×3类）
    static_mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=13, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH
    )
    delta_mfcc = librosa.feature.delta(static_mfcc)       # 一阶差分
    delta2_mfcc = librosa.feature.delta(static_mfcc, order=2)  # 二阶差分
    
    # 计算每类MFCC的统计量
    mfcc_config = [
        ("静态MFCC", static_mfcc),
        ("Delta MFCC", delta_mfcc),
        ("Delta-Delta MFCC", delta2_mfcc)
    ]
    for mfcc_type, mfcc_data in mfcc_config:
        mfcc_dim_stats = {}
        for dim_idx in range(13):
            dim_name = f"{mfcc_type}_Dim{dim_idx+1}"
            dim_stats, dim_vals = calc_statistics(mfcc_data[dim_idx], dim_name)
            mfcc_dim_stats[dim_name] = dim_stats
            freq_feats_values.extend(dim_vals)
        freq_feats_dict[mfcc_type] = mfcc_dim_stats
    
    return freq_feats_dict, freq_feats_values

# -------------------------- 特征模块3：韵律与发声音质特征 --------------------------
def extract_prosodic_phonation_features(y, sr):
    """提取：基频F0、Jitter、Shimmer、HNR、浊音段比例"""
    prosody_feats_dict = {}
    prosody_feats_values = []
    
    # 1. 基频F0（pyin属于librosa，已适配类型）
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), sr=sr)
    f0_stats, f0_vals = calc_statistics(f0, "F0")
    prosody_feats_dict["基频(F0)"] = f0_stats
    prosody_feats_values.extend(f0_vals)
    
    # 用pyworld提取声学参数（用于音质特征计算，y已转为float64）
    pw_f0, timeaxis = pw.harvest(y, sr)
    sp = pw.cheaptrick(y, pw_f0, timeaxis, sr)
    ap = pw.d4c(y, pw_f0, timeaxis, sr)
    
    # 2. 谐噪比（HNR）
    y_harmonic = pw.synthesize(pw_f0, sp, np.zeros_like(ap), sr)
    harmonic_energy = np.sum(np.square(y_harmonic))
    total_energy = np.sum(np.square(y))
    noise_energy = total_energy - harmonic_energy + 1e-8  # 避免除零
    hnr_value = 10 * np.log10(harmonic_energy / noise_energy) if harmonic_energy > 0 else 0.0
    hnr_seq = np.array([hnr_value])
    hnr_stats, hnr_vals = calc_statistics(hnr_seq, "HNR")
    prosody_feats_dict["谐噪比(HNR)"] = hnr_stats
    prosody_feats_values.extend(hnr_vals)
    
    # 3. 基频微扰（Jitter）
    voiced_f0 = pw_f0[pw_f0 > 0]
    jitter_seq = np.abs(np.diff(voiced_f0)) / (np.mean(voiced_f0) + 1e-8) if len(voiced_f0) > 1 else np.array([0.0])
    jitter_stats, jitter_vals = calc_statistics(jitter_seq, "Jitter")
    prosody_feats_dict["基频微扰(Jitter)"] = jitter_stats
    prosody_feats_values.extend(jitter_vals)
    
    # 4. 振幅微扰（Shimmer）
    frame_energy = np.array([
        np.sum(np.square(y[i:i+FRAME_LENGTH] * WINDOW)) 
        for i in range(0, len(y)-FRAME_LENGTH, HOP_LENGTH)
    ])
    shimmer_seq = np.abs(np.diff(frame_energy)) / (np.mean(frame_energy) + 1e-8) if len(frame_energy) > 1 else np.array([0.0])
    shimmer_stats, shimmer_vals = calc_statistics(shimmer_seq, "Shimmer")
    prosody_feats_dict["振幅微扰(Shimmer)"] = shimmer_stats
    prosody_feats_values.extend(shimmer_vals)
    
    # 5. 浊音段比例
    voiced_frames = pw_f0 > 0
    voiced_ratio = np.sum(voiced_frames) / len(pw_f0) if len(pw_f0) > 0 else 0.0
    voiced_ratio_seq = np.array([voiced_ratio])
    vr_stats, vr_vals = calc_statistics(voiced_ratio_seq, "VoicedRatio")
    prosody_feats_dict["浊音段比例(VoicedRatio)"] = vr_stats
    prosody_feats_values.extend(vr_vals)
    
    return prosody_feats_dict, prosody_feats_values

# -------------------------- 特征模块4：话语停顿与时序特征 --------------------------
def extract_pause_temporal_features(y, sr):
    """提取：总停顿次数、平均停顿时长、总停顿时长占比"""
    pause_feats_dict = {}
    pause_feats_values = []
    
    # 基于能量阈值的停顿检测
    frame_energy = np.array([
        np.sum(np.square(y[i:i+FRAME_LENGTH] * WINDOW)) 
        for i in range(0, len(y)-FRAME_LENGTH, HOP_LENGTH)
    ])
    energy_threshold = np.mean(frame_energy) * 0.1 if len(frame_energy) > 0 else 0.0
    pause_frame_mask = frame_energy < energy_threshold
    
    # 合并连续停顿帧，统计停顿区间
    pause_intervals = []
    in_pause = False
    pause_start = 0
    for idx, is_pause in enumerate(pause_frame_mask):
        if is_pause and not in_pause:
            in_pause = True
            pause_start = idx
        elif not is_pause and in_pause:
            in_pause = False
            pause_intervals.append((pause_start, idx))
    if in_pause:  # 处理音频末尾的停顿
        pause_intervals.append((pause_start, len(pause_frame_mask)))
    
    total_duration = librosa.get_duration(y=y, sr=sr)
    total_pause_count = len(pause_intervals)
    
    # 1. 总停顿次数
    pc_stats, pc_vals = calc_statistics(np.array([total_pause_count]), "TotalPauseCount")
    pause_feats_dict["总停顿次数"] = pc_stats
    pause_feats_values.extend(pc_vals)
    
    # 2. 平均停顿时长
    if total_pause_count > 0:
        pause_durations = [(end - start) * (HOP_LENGTH / sr) for (start, end) in pause_intervals]
        avg_pause = np.mean(pause_durations)
    else:
        avg_pause = 0.0
    ap_stats, ap_vals = calc_statistics(np.array([avg_pause]), "AvgPauseDuration")
    pause_feats_dict["平均停顿时长(s)"] = ap_stats
    pause_feats_values.extend(ap_vals)
    
    # 3. 总停顿时长占比
    if total_pause_count > 0 and total_duration > 0:
        total_pause_dur = np.sum([(end - start) * (HOP_LENGTH / sr) for (start, end) in pause_intervals])
        pause_ratio = total_pause_dur / total_duration
    else:
        pause_ratio = 0.0
    pr_stats, pr_vals = calc_statistics(np.array([pause_ratio]), "PauseDurationRatio")
    pause_feats_dict["总停顿时长占比"] = pr_stats
    pause_feats_values.extend(pr_vals)
    
    return pause_feats_dict, pause_feats_values

# -------------------------- 主函数：特征提取入口 --------------------------
def extract_suicide_risk_acoustic_features(audio_path, return_detail_stats=True):
    """
    自杀风险语音声学特征提取主函数
    :param audio_path: 音频文件路径
    :param return_detail_stats: 是否返回分类别详细统计
    :return: 标准化特征向量 + 详细统计字典（可选）
    """
    # 1. 音频预处理
    y, sr = preprocess_audio(audio_path)
    
    # 2. 分模块提取四大类特征
    time_dict, time_vals = extract_time_domain_features(y, sr)
    freq_dict, freq_vals = extract_frequency_cepstral_features(y, sr)
    prosody_dict, prosody_vals = extract_prosodic_phonation_features(y, sr)
    pause_dict, pause_vals = extract_pause_temporal_features(y, sr)
    
    # 3. 拼接所有特征值并标准化
    all_raw_feats = np.array(time_vals + freq_vals + prosody_vals + pause_vals)
    mean_val = np.mean(all_raw_feats)
    std_val = np.std(all_raw_feats)
    
    # 避免标准差为0的边界情况
    if std_val != 0:
        final_feats = (all_raw_feats - mean_val) / std_val
    else:
        final_feats = all_raw_feats - mean_val
    
    # 4. 整理详细统计结果（用于论文分析）
    if return_detail_stats:
        detail_dict = {
            "1. 时域声学特征": time_dict,
            "2. 频域与倒谱特征": freq_dict,
            "3. 韵律与发声音质特征": prosody_dict,
            "4. 话语停顿与时序特征": pause_dict,
            "全局汇总": {
                "原始特征总维度": len(all_raw_feats),
                "标准化后均值": float(np.mean(final_feats)),
                "标准化后标准差": float(np.std(final_feats))
            }
        }
        return final_feats, detail_dict
    return final_feats

# -------------------------- 测试示例 --------------------------
if __name__ == "__main__":
    # 替换为你的音频文件路径
    TEST_AUDIO_PATH = "/home/user512-003/songcw/code/scw-ser/data/wav/Ses01F_impro01/Ses01F_impro01_F000.wav"
    
    # 提取特征
    final_features, detail_stats = extract_suicide_risk_acoustic_features(
        TEST_AUDIO_PATH, return_detail_stats=True
    )
    
    # 打印核心结果
    print("="*90)
    print("✅ 自杀风险语音声学特征提取完成")
    print("="*90)
    print(f"最终特征维度：{final_features.shape[0]} (预期373维)")
    print(f"标准化后全局均值：{detail_stats['全局汇总']['标准化后均值']:.6f}")
    print(f"标准化后全局标准差：{detail_stats['全局汇总']['标准化后标准差']:.6f}")
    print(f"\n前10维特征值：{final_features[:10]}")
    
    # 可选：打印分类别详细统计（取消注释查看）
    # import json
    # print("\n" + "="*90)
    # print("📊 分类别详细统计数据")
    # print("="*90)
    # print(json.dumps(detail_stats, indent=2, ensure_ascii=False))

✅ 自杀风险语音声学特征提取完成
最终特征维度：301 (预期373维)
标准化后全局均值：nan
标准化后全局标准差：nan

前10维特征值：[nan nan nan nan nan nan nan nan nan nan]


/tmp/ipykernel_121137/3509951305.py:156: RuntimeWarning: invalid value encountered in log10
  hnr_value = 10 * np.log10(harmonic_energy / noise_energy) if harmonic_energy > 0 else 0.0


In [11]:
import librosa
import numpy as np
from scipy import stats
import noisereduce as nr
import pyworld as pw
import warnings
import json

# 过滤可选警告（不影响核心功能）
warnings.filterwarnings("ignore", message="IProgress not found. Please update jupyter and ipywidgets")
warnings.filterwarnings("ignore", message="PySoundFile failed. Trying audioread instead.")

# -------------------------- 全局配置 --------------------------
# 1. 特征提取参数
TARGET_SR = 16000  # 目标采样率
FRAME_LENGTH = int(TARGET_SR * 0.025)  # 25ms帧长
HOP_LENGTH = int(TARGET_SR * 0.01)     # 10ms帧移
WINDOW = np.hamming(FRAME_LENGTH)      # 汉明窗（减少频谱泄漏）

# 2. 特征类别学术说明（适配SCI论文）
FEATURE_CATEGORY_DESC = {
    "time_domain": {
        "name_cn": "时域声学特征",
        "name_en": "Time-domain Acoustic Features",
        "sub_features": ["过零率(ZCR)", "短时能量(ShortTermEnergy)", "语速(SpeechRate)"],
        "clinical_significance": "反映语音的底层发声强度、节奏波动，自杀/抑郁高风险人群常表现为能量偏低、语速异常（过快/过慢/停顿多）",
        "dimension_calc": "6(ZCR) + 6(短时能量) + 1(语速) = 13维"
    },
    "frequency_cepstral": {
        "name_cn": "频域与倒谱特征",
        "name_en": "Frequency-domain & Cepstral Features",
        "sub_features": ["频谱质心(SpectralCentroid)", "静态MFCC(13维)", "Delta MFCC(13维)", "Delta-Delta MFCC(13维)"],
        "clinical_significance": "反映语音的频谱分布、音色与动态变化，捕捉抑郁/自杀人群语调平淡、频谱复杂度低的核心特征",
        "dimension_calc": "6(频谱质心) + 13×6(静态MFCC) + 13×6(Delta MFCC) + 13×6(Delta-Delta MFCC) = 249维"
    },
    "prosodic_phonation": {
        "name_cn": "韵律与发声音质特征",
        "name_en": "Prosodic & Phonation Quality Features",
        "sub_features": ["基频(F0)", "谐噪比(HNR)", "基频微扰(Jitter)", "振幅微扰(Shimmer)", "浊音段比例(VoicedRatio)"],
        "clinical_significance": "自杀风险最核心的声学标志物，高风险人群常表现为F0范围收窄、Jitter/Shimmer升高、HNR降低（声带振动不稳定）",
        "dimension_calc": "6(F0) + 6(HNR) + 6(Jitter) + 6(Shimmer) + 6(VoicedRatio) = 30维"  # 修正：原代码中是5个子特征×6=30维
    },
    "pause_temporal": {
        "name_cn": "话语停顿与时序特征",
        "name_en": "Utterance Pause & Temporal Features",
        "sub_features": ["总停顿次数(TotalPauseCount)", "平均停顿时长(AvgPauseDuration)", "总停顿时长占比(PauseDurationRatio)"],
        "clinical_significance": "反映言语流畅性、思维连贯性，高风险人群常表现为停顿次数多、停顿时长占比>30%、思维碎片化",
        "dimension_calc": "6(总停顿次数) + 6(平均停顿时长) + 6(总停顿时长占比) = 18维"
    },
    "total": {
        "name_cn": "全局特征汇总",
        "dimension_calc": "13(时域) + 249(频域) + 30(韵律) + 18(停顿) = 310维",  # 修正实际维度：13+249+30+18=310
        "note": "维度修正说明：原373维为笔误，实际子特征统计为310维，核心特征体系不变"
    }
}

# -------------------------- 通用工具函数 --------------------------
def calc_statistics(feat, feat_name):
    """
    计算单个特征序列的6类核心统计量（均值/标准差/最大值/最小值/偏度/峰度）
    兼容空值/NaN值处理，避免计算报错
    """
    # 处理空特征/全NaN特征的边界情况
    if len(feat) == 0 or np.all(np.isnan(feat)):
        stats_dict = {
            f"{feat_name}_mean": 0.0,
            f"{feat_name}_std": 0.0,
            f"{feat_name}_max": 0.0,
            f"{feat_name}_min": 0.0,
            f"{feat_name}_skewness": 0.0,
            f"{feat_name}_kurtosis": 0.0
        }
        return stats_dict, list(stats_dict.values())
    
    # 去除有效序列中的NaN值
    feat = feat[~np.isnan(feat)]
    
    # 计算6类统计量
    stats_dict = {
        f"{feat_name}_mean": round(float(np.mean(feat)), 6),
        f"{feat_name}_std": round(float(np.std(feat)), 6),
        f"{feat_name}_max": round(float(np.max(feat)), 6),
        f"{feat_name}_min": round(float(np.min(feat)), 6),
        f"{feat_name}_skewness": round(float(stats.skew(feat)), 6),
        f"{feat_name}_kurtosis": round(float(stats.kurtosis(feat)), 6)
    }
    return stats_dict, list(stats_dict.values())

def print_feature_detail_info(feature_data, category_key):
    """
    打印单类特征的详细说明与统计信息
    :param feature_data: 特征统计字典
    :param category_key: 特征类别键（对应FEATURE_CATEGORY_DESC）
    """
    desc = FEATURE_CATEGORY_DESC[category_key]
    print(f"\n{'='*100}")
    print(f"📋 特征类别：{desc['name_cn']} ({desc['name_en']})")
    print(f"🔍 临床意义：{desc['clinical_significance']}")
    print(f"📏 维度构成：{desc['dimension_calc']}")
    print(f"🧩 包含子特征：{', '.join(desc['sub_features'])}")
    print(f"\n📊 详细统计值：")
    print("-"*80)
    
    # 格式化输出每个子特征的统计值
    for sub_feat_name, sub_feat_stats in feature_data.items():
        print(f"\n【{sub_feat_name}】")
        if isinstance(sub_feat_stats, dict):
            # 6类统计量的子特征
            for stat_name, stat_value in sub_feat_stats.items():
                stat_type = stat_name.split('_')[-1]
                stat_cn = {
                    'mean': '均值', 'std': '标准差', 'max': '最大值',
                    'min': '最小值', 'skewness': '偏度', 'kurtosis': '峰度',
                    'value': '数值'
                }.get(stat_type, stat_type)
                print(f"  {stat_cn:6s}：{stat_value}")
        else:
            # 特殊值（如语速）
            print(f"  数值     ：{sub_feat_stats}")

# -------------------------- 音频预处理 --------------------------
def preprocess_audio(audio_path):
    """
    标准化音频预处理：统一采样率 + 专业去噪 + 类型转换为float64（适配pyworld）
    """
    # 读取音频并统一重采样至目标采样率
    y, orig_sr = librosa.load(audio_path, sr=None)
    y = librosa.resample(y, orig_sr=orig_sr, target_sr=TARGET_SR)
    
    # 核心修复：将音频信号转为float64（double），适配pyworld的类型要求
    y = y.astype(np.float64)
    
    # 基于noisereduce的语音去噪（保留有效发声信息）
    y = nr.reduce_noise(y=y, sr=TARGET_SR)
    
    return y, TARGET_SR

# -------------------------- 特征提取模块 --------------------------
def extract_time_domain_features(y, sr):
    """提取：过零率(ZCR)、短时能量、语速"""
    time_feats_dict = {}
    time_feats_values = []
    
    # 1. 过零率（ZCR）
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]
    zcr_stats, zcr_vals = calc_statistics(zcr, "ZCR")
    time_feats_dict["过零率(ZCR)"] = zcr_stats
    time_feats_values.extend(zcr_vals)
    
    # 2. 短时能量
    short_term_energy = np.array([
        np.sum(np.square(y[i:i+FRAME_LENGTH] * WINDOW)) 
        for i in range(0, len(y)-FRAME_LENGTH, HOP_LENGTH)
    ])
    energy_stats, energy_vals = calc_statistics(short_term_energy, "ShortTermEnergy")
    time_feats_dict["短时能量(ShortTermEnergy)"] = energy_stats
    time_feats_values.extend(energy_vals)
    
    # 3. 语速（帧数量/音频总时长）
    total_duration = librosa.get_duration(y=y, sr=sr)
    speech_rate = len(zcr) / total_duration if (len(y) > 0 and total_duration > 0) else 0.0
    time_feats_dict["语速(SpeechRate)"] = {"SpeechRate_value": round(float(speech_rate), 6)}
    time_feats_values.append(float(speech_rate))
    
    return time_feats_dict, time_feats_values

def extract_frequency_cepstral_features(y, sr):
    """提取：频谱质心、静态MFCC、Delta MFCC、Delta-Delta MFCC"""
    freq_feats_dict = {}
    freq_feats_values = []
    
    # 1. 频谱质心
    spectral_centroid = librosa.feature.spectral_centroid(
        y=y, sr=sr, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH
    )[0]
    sc_stats, sc_vals = calc_statistics(spectral_centroid, "SpectralCentroid")
    freq_feats_dict["频谱质心(SpectralCentroid)"] = sc_stats
    freq_feats_values.extend(sc_vals)
    
    # 2. 静态MFCC + 动态差分MFCC（13维×3类）
    static_mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=13, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH
    )
    delta_mfcc = librosa.feature.delta(static_mfcc)       # 一阶差分
    delta2_mfcc = librosa.feature.delta(static_mfcc, order=2)  # 二阶差分
    
    # 计算每类MFCC的统计量
    mfcc_config = [
        ("静态MFCC", static_mfcc),
        ("Delta MFCC", delta_mfcc),
        ("Delta-Delta MFCC", delta2_mfcc)
    ]
    for mfcc_type, mfcc_data in mfcc_config:
        mfcc_dim_stats = {}
        for dim_idx in range(13):
            dim_name = f"{mfcc_type}_Dim{dim_idx+1}"
            dim_stats, dim_vals = calc_statistics(mfcc_data[dim_idx], dim_name)
            mfcc_dim_stats[dim_name] = dim_stats
            freq_feats_values.extend(dim_vals)
        freq_feats_dict[mfcc_type] = mfcc_dim_stats
    
    return freq_feats_dict, freq_feats_values

def extract_prosodic_phonation_features(y, sr):
    """提取：基频F0、Jitter、Shimmer、HNR、浊音段比例"""
    prosody_feats_dict = {}
    prosody_feats_values = []
    
    # 1. 基频F0（pyin属于librosa，已适配类型）
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), sr=sr)
    f0_stats, f0_vals = calc_statistics(f0, "F0")
    prosody_feats_dict["基频(F0)"] = f0_stats
    prosody_feats_values.extend(f0_vals)
    
    # 用pyworld提取声学参数（用于音质特征计算，y已转为float64）
    pw_f0, timeaxis = pw.harvest(y, sr)
    sp = pw.cheaptrick(y, pw_f0, timeaxis, sr)
    ap = pw.d4c(y, pw_f0, timeaxis, sr)
    
    # 2. 谐噪比（HNR）
    y_harmonic = pw.synthesize(pw_f0, sp, np.zeros_like(ap), sr)
    harmonic_energy = np.sum(np.square(y_harmonic))
    total_energy = np.sum(np.square(y))
    noise_energy = total_energy - harmonic_energy + 1e-8  # 避免除零
    hnr_value = 10 * np.log10(harmonic_energy / noise_energy) if harmonic_energy > 0 else 0.0
    hnr_seq = np.array([hnr_value])
    hnr_stats, hnr_vals = calc_statistics(hnr_seq, "HNR")
    prosody_feats_dict["谐噪比(HNR)"] = hnr_stats
    prosody_feats_values.extend(hnr_vals)
    
    # 3. 基频微扰（Jitter）
    voiced_f0 = pw_f0[pw_f0 > 0]
    jitter_seq = np.abs(np.diff(voiced_f0)) / (np.mean(voiced_f0) + 1e-8) if len(voiced_f0) > 1 else np.array([0.0])
    jitter_stats, jitter_vals = calc_statistics(jitter_seq, "Jitter")
    prosody_feats_dict["基频微扰(Jitter)"] = jitter_stats
    prosody_feats_values.extend(jitter_vals)
    
    # 4. 振幅微扰（Shimmer）
    frame_energy = np.array([
        np.sum(np.square(y[i:i+FRAME_LENGTH] * WINDOW)) 
        for i in range(0, len(y)-FRAME_LENGTH, HOP_LENGTH)
    ])
    shimmer_seq = np.abs(np.diff(frame_energy)) / (np.mean(frame_energy) + 1e-8) if len(frame_energy) > 1 else np.array([0.0])
    shimmer_stats, shimmer_vals = calc_statistics(shimmer_seq, "Shimmer")
    prosody_feats_dict["振幅微扰(Shimmer)"] = shimmer_stats
    prosody_feats_values.extend(shimmer_vals)
    
    # 5. 浊音段比例
    voiced_frames = pw_f0 > 0
    voiced_ratio = np.sum(voiced_frames) / len(pw_f0) if len(pw_f0) > 0 else 0.0
    voiced_ratio_seq = np.array([voiced_ratio])
    vr_stats, vr_vals = calc_statistics(voiced_ratio_seq, "VoicedRatio")
    prosody_feats_dict["浊音段比例(VoicedRatio)"] = vr_stats
    prosody_feats_values.extend(vr_vals)
    
    return prosody_feats_dict, prosody_feats_values

def extract_pause_temporal_features(y, sr):
    """提取：总停顿次数、平均停顿时长、总停顿时长占比"""
    pause_feats_dict = {}
    pause_feats_values = []
    
    # 基于能量阈值的停顿检测
    frame_energy = np.array([
        np.sum(np.square(y[i:i+FRAME_LENGTH] * WINDOW)) 
        for i in range(0, len(y)-FRAME_LENGTH, HOP_LENGTH)
    ])
    energy_threshold = np.mean(frame_energy) * 0.1 if len(frame_energy) > 0 else 0.0
    pause_frame_mask = frame_energy < energy_threshold
    
    # 合并连续停顿帧，统计停顿区间
    pause_intervals = []
    in_pause = False
    pause_start = 0
    for idx, is_pause in enumerate(pause_frame_mask):
        if is_pause and not in_pause:
            in_pause = True
            pause_start = idx
        elif not is_pause and in_pause:
            in_pause = False
            pause_intervals.append((pause_start, idx))
    if in_pause:  # 处理音频末尾的停顿
        pause_intervals.append((pause_start, len(pause_frame_mask)))
    
    total_duration = librosa.get_duration(y=y, sr=sr)
    total_pause_count = len(pause_intervals)
    
    # 1. 总停顿次数
    pc_stats, pc_vals = calc_statistics(np.array([total_pause_count]), "TotalPauseCount")
    pause_feats_dict["总停顿次数(TotalPauseCount)"] = pc_stats
    pause_feats_values.extend(pc_vals)
    
    # 2. 平均停顿时长
    if total_pause_count > 0:
        pause_durations = [(end - start) * (HOP_LENGTH / sr) for (start, end) in pause_intervals]
        avg_pause = np.mean(pause_durations)
    else:
        avg_pause = 0.0
    ap_stats, ap_vals = calc_statistics(np.array([avg_pause]), "AvgPauseDuration")
    pause_feats_dict["平均停顿时长(AvgPauseDuration)"] = ap_stats
    pause_feats_values.extend(ap_vals)
    
    # 3. 总停顿时长占比
    if total_pause_count > 0 and total_duration > 0:
        total_pause_dur = np.sum([(end - start) * (HOP_LENGTH / sr) for (start, end) in pause_intervals])
        pause_ratio = total_pause_dur / total_duration
    else:
        pause_ratio = 0.0
    pr_stats, pr_vals = calc_statistics(np.array([pause_ratio]), "PauseDurationRatio")
    pause_feats_dict["总停顿时长占比(PauseDurationRatio)"] = pr_stats
    pause_feats_values.extend(pr_vals)
    
    return pause_feats_dict, pause_feats_values

# -------------------------- 主函数：特征提取入口 --------------------------
def extract_suicide_risk_acoustic_features(audio_path, return_detail_stats=True):
    """
    自杀风险语音声学特征提取主函数
    :param audio_path: 音频文件路径
    :param return_detail_stats: 是否返回分类别详细统计
    :return: 标准化特征向量 + 详细统计字典（可选）
    """
    # 1. 音频预处理
    y, sr = preprocess_audio(audio_path)
    
    # 2. 分模块提取四大类特征
    time_dict, time_vals = extract_time_domain_features(y, sr)
    freq_dict, freq_vals = extract_frequency_cepstral_features(y, sr)
    prosody_dict, prosody_vals = extract_prosodic_phonation_features(y, sr)
    pause_dict, pause_vals = extract_pause_temporal_features(y, sr)
    
    # 3. 拼接所有特征值并标准化
    all_raw_feats = np.array(time_vals + freq_vals + prosody_vals + pause_vals)
    mean_val = np.mean(all_raw_feats)
    std_val = np.std(all_raw_feats)
    
    # 避免标准差为0的边界情况
    if std_val != 0:
        final_feats = (all_raw_feats - mean_val) / std_val
    else:
        final_feats = all_raw_feats - mean_val
    
    # 4. 整理详细统计结果（用于论文分析）
    if return_detail_stats:
        detail_dict = {
            "time_domain": time_dict,
            "frequency_cepstral": freq_dict,
            "prosodic_phonation": prosody_dict,
            "pause_temporal": pause_dict,
            "global_summary": {
                "原始特征总维度": len(all_raw_feats),
                "标准化后均值": round(float(np.mean(final_feats)), 6),
                "标准化后标准差": round(float(np.std(final_feats)), 6),
                "各维度明细": FEATURE_CATEGORY_DESC["total"]["dimension_calc"],
                "维度说明": FEATURE_CATEGORY_DESC["total"]["note"]
            }
        }
        return final_feats, detail_dict
    return final_feats

# -------------------------- 测试与详细输出 --------------------------
if __name__ == "__main__":
    # 替换为你的音频文件路径
    TEST_AUDIO_PATH = "/home/user512-003/songcw/code/scw-ser/data/wav/Ses01F_impro01/Ses01F_impro01_F000.wav"
    
    # 提取特征
    print("🔄 开始提取自杀风险语音声学特征...")
    final_features, detail_stats = extract_suicide_risk_acoustic_features(
        TEST_AUDIO_PATH, return_detail_stats=True
    )
    
    # 1. 打印全局汇总信息
    print(f"\n{'='*100}")
    print("✅ 特征提取完成 - 全局汇总信息")
    print(f"{'='*100}")
    global_summary = detail_stats["global_summary"]
    print(f"📈 最终标准化特征维度：{global_summary['原始特征总维度']}")
    print(f"📊 标准化后全局均值：{global_summary['标准化后均值']}")
    print(f"📊 标准化后全局标准差：{global_summary['标准化后标准差']}")
    print(f"📏 维度构成明细：{global_summary['各维度明细']}")
    print(f"💡 维度说明：{global_summary['维度说明']}")
    
    # 2. 打印每类特征的详细说明与统计
    # 2.1 时域声学特征
    print_feature_detail_info(detail_stats["time_domain"], "time_domain")
    
    # 2.2 韵律与发声音质特征（核心临床特征，优先展示）
    print_feature_detail_info(detail_stats["prosodic_phonation"], "prosodic_phonation")
    
    # 2.3 话语停顿与时序特征
    print_feature_detail_info(detail_stats["pause_temporal"], "pause_temporal")
    
    # 2.4 频域与倒谱特征（维度较多，可选展示）
    print(f"\n{'='*100}")
    print(f"📋 特征类别：{FEATURE_CATEGORY_DESC['frequency_cepstral']['name_cn']} ({FEATURE_CATEGORY_DESC['frequency_cepstral']['name_en']})")
    print(f"🔍 临床意义：{FEATURE_CATEGORY_DESC['frequency_cepstral']['clinical_significance']}")
    print(f"📏 维度构成：{FEATURE_CATEGORY_DESC['frequency_cepstral']['dimension_calc']}")
    print(f"💡 提示：频域特征维度较多（240维），如需查看完整统计值，请取消下方注释")
    # 取消以下注释可查看频域特征完整统计
    # print(f"\n📊 详细统计值：")
    # print(json.dumps(detail_stats["frequency_cepstral"], indent=2, ensure_ascii=False))
    
    # 3. 导出详细统计结果到JSON文件（方便论文写作/后续分析）
    export_data = {
        "feature_category_description": FEATURE_CATEGORY_DESC,
        "global_summary": global_summary,
        "detailed_statistics": {
            "时域声学特征": detail_stats["time_domain"],
            "韵律与发声音质特征": detail_stats["prosodic_phonation"],
            "话语停顿与时序特征": detail_stats["pause_temporal"],
            "频域与倒谱特征": "详见代码内输出（维度较多）"
        }
    }
    with open("suicide_risk_audio_features_stats.json", "w", encoding="utf-8") as f:
        json.dump(export_data, f, ensure_ascii=False, indent=2)
    print(f"\n💾 详细统计结果已导出至：suicide_risk_audio_features_stats.json")
    print(f"{'='*100}")

🔄 开始提取自杀风险语音声学特征...

✅ 特征提取完成 - 全局汇总信息
📈 最终标准化特征维度：301
📊 标准化后全局均值：nan
📊 标准化后全局标准差：nan
📏 维度构成明细：13(时域) + 249(频域) + 30(韵律) + 18(停顿) = 310维
💡 维度说明：维度修正说明：原373维为笔误，实际子特征统计为310维，核心特征体系不变

📋 特征类别：时域声学特征 (Time-domain Acoustic Features)
🔍 临床意义：反映语音的底层发声强度、节奏波动，自杀/抑郁高风险人群常表现为能量偏低、语速异常（过快/过慢/停顿多）
📏 维度构成：6(ZCR) + 6(短时能量) + 1(语速) = 13维
🧩 包含子特征：过零率(ZCR), 短时能量(ShortTermEnergy), 语速(SpeechRate)

📊 详细统计值：
--------------------------------------------------------------------------------

【过零率(ZCR)】
  均值    ：0.144346
  标准差   ：0.15778
  最大值   ：0.725
  最小值   ：0.01
  偏度    ：2.050004
  峰度    ：3.550289

【短时能量(ShortTermEnergy)】
  均值    ：0.00611
  标准差   ：0.017299
  最大值   ：0.093842
  最小值   ：2e-06
  偏度    ：3.309787
  峰度    ：10.838364

【语速(SpeechRate)】
  数值    ：100.228083

📋 特征类别：韵律与发声音质特征 (Prosodic & Phonation Quality Features)
🔍 临床意义：自杀风险最核心的声学标志物，高风险人群常表现为F0范围收窄、Jitter/Shimmer升高、HNR降低（声带振动不稳定）
📏 维度构成：6(F0) + 6(HNR) + 6(Jitter) + 6(Shimmer) + 6(VoicedRatio) = 30维
🧩 包含子特征：基频(F0), 谐噪比(HNR), 基频微扰(Jitter), 振幅微扰(Shimm

/tmp/ipykernel_121137/1047544372.py:225: RuntimeWarning: invalid value encountered in log10
  hnr_value = 10 * np.log10(harmonic_energy / noise_energy) if harmonic_energy > 0 else 0.0


In [13]:
import librosa
import numpy as np
from scipy import stats
import noisereduce as nr
import pyworld as pw
import warnings
import json

# 过滤可选警告（不影响核心功能）
warnings.filterwarnings("ignore", message="IProgress not found. Please update jupyter and ipywidgets")
warnings.filterwarnings("ignore", message="PySoundFile failed. Trying audioread instead.")

# -------------------------- 全局配置 --------------------------
TARGET_SR = 16000  # 目标采样率
FRAME_LENGTH = int(TARGET_SR * 0.025)  # 25ms帧长
HOP_LENGTH = int(TARGET_SR * 0.01)     # 10ms帧移
WINDOW = np.hamming(FRAME_LENGTH)      # 汉明窗（减少频谱泄漏）

# 特征类别学术说明（适配SCI论文）
FEATURE_CATEGORY_DESC = {
    "time_domain": {
        "name_cn": "时域声学特征",
        "name_en": "Time-domain Acoustic Features",
        "sub_features": ["过零率(ZCR)", "短时能量(ShortTermEnergy)", "语速(SpeechRate)"],
        "clinical_significance": "反映语音的底层发声强度、节奏波动，自杀/抑郁高风险人群常表现为能量偏低、语速异常（过快/过慢/停顿多）",
        "dimension_calc": "6(ZCR) + 6(短时能量) + 1(语速) = 13维"
    },
    "frequency_cepstral": {
        "name_cn": "频域与倒谱特征",
        "name_en": "Frequency-domain & Cepstral Features",
        "sub_features": ["频谱质心(SpectralCentroid)", "静态MFCC(13维)", "Delta MFCC(13维)", "Delta-Delta MFCC(13维)"],
        "clinical_significance": "反映语音的频谱分布、音色与动态变化，捕捉抑郁/自杀人群语调平淡、频谱复杂度低的核心特征",
        "dimension_calc": "6(频谱质心) + 13×6(静态MFCC) + 13×6(Delta MFCC) + 13×6(Delta-Delta MFCC) = 240维"
    },
    "prosodic_phonation": {
        "name_cn": "韵律与发声音质特征",
        "name_en": "Prosodic & Phonation Quality Features",
        "sub_features": ["基频(F0)", "谐噪比(HNR)", "基频微扰(Jitter)", "振幅微扰(Shimmer)", "浊音段比例(VoicedRatio)"],
        "clinical_significance": "自杀风险最核心的声学标志物，高风险人群常表现为F0范围收窄、Jitter/Shimmer升高、HNR降低（声带振动不稳定）",
        "dimension_calc": "6(F0) + 6(HNR) + 6(Jitter) + 6(Shimmer) + 6(VoicedRatio) = 30维"
    },
    "pause_temporal": {
        "name_cn": "话语停顿与时序特征",
        "name_en": "Utterance Pause & Temporal Features",
        "sub_features": ["总停顿次数(TotalPauseCount)", "平均停顿时长(AvgPauseDuration)", "总停顿时长占比(PauseDurationRatio)"],
        "clinical_significance": "反映言语流畅性、思维连贯性，高风险人群常表现为停顿次数多、停顿时长占比>30%、思维碎片化",
        "dimension_calc": "6(总停顿次数) + 6(平均停顿时长) + 6(总停顿时长占比) = 18维"
    },
    "total": {
        "name_cn": "全局特征汇总",
        "dimension_calc": "13(时域) + 240(频域) + 30(韵律) + 18(停顿) = 301维",
        "note": "维度修正说明：原373维为笔误，实际子特征统计为301维，核心特征体系不变"
    }
}

# -------------------------- 核心优化：统计量计算函数 --------------------------
def calc_statistics(feat, feat_name):
    """
    计算单个特征序列的6类核心统计量（均值/标准差/最大值/最小值/偏度/峰度）
    新增：有效数据长度判断 + 方差为0处理，避免偏度/峰度出现nan
    """
    # 步骤1：处理空特征/全NaN特征
    if len(feat) == 0 or np.all(np.isnan(feat)):
        return {
            f"{feat_name}_mean": 0.0,
            f"{feat_name}_std": 0.0,
            f"{feat_name}_max": 0.0,
            f"{feat_name}_min": 0.0,
            f"{feat_name}_skewness": 0.0,
            f"{feat_name}_kurtosis": 0.0
        }, [0.0]*6
    
    # 步骤2：过滤NaN值，保留有效数据
    valid_feat = feat[~np.isnan(feat)]
    
    # 步骤3：判断有效数据长度（偏度/峰度需至少3个不同数据点）
    if len(valid_feat) < 3:
        mean_val = round(float(np.mean(valid_feat)), 6)
        std_val = round(float(np.std(valid_feat)), 6)
        max_val = round(float(np.max(valid_feat)), 6)
        min_val = round(float(np.min(valid_feat)), 6)
        return {
            f"{feat_name}_mean": mean_val,
            f"{feat_name}_std": std_val,
            f"{feat_name}_max": max_val,
            f"{feat_name}_min": min_val,
            f"{feat_name}_skewness": 0.0,  # 数据不足时默认0（学术上可接受）
            f"{feat_name}_kurtosis": 0.0
        }, [mean_val, std_val, max_val, min_val, 0.0, 0.0]
    
    # 步骤4：计算基础统计量
    mean_val = round(float(np.mean(valid_feat)), 6)
    std_val = round(float(np.std(valid_feat)), 6)
    max_val = round(float(np.max(valid_feat)), 6)
    min_val = round(float(np.min(valid_feat)), 6)
    
    # 步骤5：处理方差为0（所有有效数据值相同）
    if std_val < 1e-8:  # 方差接近0，视为无波动
        return {
            f"{feat_name}_mean": mean_val,
            f"{feat_name}_std": 0.0,
            f"{feat_name}_max": max_val,
            f"{feat_name}_min": min_val,
            f"{feat_name}_skewness": 0.0,  # 对称分布（无偏）
            f"{feat_name}_kurtosis": 3.0   # 正态分布峰度为3，视为基准
        }, [mean_val, 0.0, max_val, min_val, 0.0, 3.0]
    
    # 步骤6：正常计算偏度和峰度（保留6位小数）
    skewness_val = round(float(stats.skew(valid_feat)), 6)
    kurtosis_val = round(float(stats.kurtosis(valid_feat)), 6)
    
    # 步骤7：返回结果
    stats_dict = {
        f"{feat_name}_mean": mean_val,
        f"{feat_name}_std": std_val,
        f"{feat_name}_max": max_val,
        f"{feat_name}_min": min_val,
        f"{feat_name}_skewness": skewness_val,
        f"{feat_name}_kurtosis": kurtosis_val
    }
    return stats_dict, list(stats_dict.values())

def print_feature_detail_info(feature_data, category_key):
    """打印单类特征的详细说明与统计信息"""
    desc = FEATURE_CATEGORY_DESC[category_key]
    print(f"\n{'='*100}")
    print(f"📋 特征类别：{desc['name_cn']} ({desc['name_en']})")
    print(f"🔍 临床意义：{desc['clinical_significance']}")
    print(f"📏 维度构成：{desc['dimension_calc']}")
    print(f"🧩 包含子特征：{', '.join(desc['sub_features'])}")
    print(f"\n📊 详细统计值：")
    print("-"*80)
    
    for sub_feat_name, sub_feat_stats in feature_data.items():
        print(f"\n【{sub_feat_name}】")
        if isinstance(sub_feat_stats, dict):
            for stat_name, stat_value in sub_feat_stats.items():
                stat_type = stat_name.split('_')[-1]
                stat_cn = {
                    'mean': '均值', 'std': '标准差', 'max': '最大值',
                    'min': '最小值', 'skewness': '偏度', 'kurtosis': '峰度',
                    'value': '数值'
                }.get(stat_type, stat_type)
                print(f"  {stat_cn:6s}：{stat_value}")
        else:
            print(f"  数值     ：{sub_feat_stats}")

# -------------------------- 音频预处理 --------------------------
def preprocess_audio(audio_path):
    """标准化音频预处理：统一采样率 + 专业去噪 + 类型转换为float64（适配pyworld）"""
    y, orig_sr = librosa.load(audio_path, sr=None)
    y = librosa.resample(y, orig_sr=orig_sr, target_sr=TARGET_SR)
    y = y.astype(np.float64)  # 适配pyworld类型要求
    y = nr.reduce_noise(y=y, sr=TARGET_SR)
    return y, TARGET_SR

# -------------------------- 特征提取模块 --------------------------
def extract_time_domain_features(y, sr):
    """提取：过零率(ZCR)、短时能量、语速"""
    time_feats_dict = {}
    time_feats_values = []
    
    # 1. 过零率（ZCR）
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]
    zcr_stats, zcr_vals = calc_statistics(zcr, "ZCR")
    time_feats_dict["过零率(ZCR)"] = zcr_stats
    time_feats_values.extend(zcr_vals)
    
    # 2. 短时能量
    short_term_energy = np.array([
        np.sum(np.square(y[i:i+FRAME_LENGTH] * WINDOW)) 
        for i in range(0, len(y)-FRAME_LENGTH, HOP_LENGTH)
    ])
    energy_stats, energy_vals = calc_statistics(short_term_energy, "ShortTermEnergy")
    time_feats_dict["短时能量(ShortTermEnergy)"] = energy_stats
    time_feats_values.extend(energy_vals)
    
    # 3. 语速（帧数量/音频总时长）
    total_duration = librosa.get_duration(y=y, sr=sr)
    speech_rate = len(zcr) / total_duration if (len(y) > 0 and total_duration > 0) else 0.0
    time_feats_dict["语速(SpeechRate)"] = {"SpeechRate_value": round(float(speech_rate), 6)}
    time_feats_values.append(float(speech_rate))
    
    return time_feats_dict, time_feats_values

def extract_frequency_cepstral_features(y, sr):
    """提取：频谱质心、静态MFCC、Delta MFCC、Delta-Delta MFCC"""
    freq_feats_dict = {}
    freq_feats_values = []
    
    # 1. 频谱质心
    spectral_centroid = librosa.feature.spectral_centroid(
        y=y, sr=sr, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH
    )[0]
    sc_stats, sc_vals = calc_statistics(spectral_centroid, "SpectralCentroid")
    freq_feats_dict["频谱质心(SpectralCentroid)"] = sc_stats
    freq_feats_values.extend(sc_vals)
    
    # 2. 静态MFCC + 动态差分MFCC（13维×3类）
    static_mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=13, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH
    )
    delta_mfcc = librosa.feature.delta(static_mfcc)       # 一阶差分
    delta2_mfcc = librosa.feature.delta(static_mfcc, order=2)  # 二阶差分
    
    mfcc_config = [("静态MFCC", static_mfcc), ("Delta MFCC", delta_mfcc), ("Delta-Delta MFCC", delta2_mfcc)]
    for mfcc_type, mfcc_data in mfcc_config:
        mfcc_dim_stats = {}
        for dim_idx in range(13):
            dim_name = f"{mfcc_type}_Dim{dim_idx+1}"
            dim_stats, dim_vals = calc_statistics(mfcc_data[dim_idx], dim_name)
            mfcc_dim_stats[dim_name] = dim_stats
            freq_feats_values.extend(dim_vals)
        freq_feats_dict[mfcc_type] = mfcc_dim_stats
    
    return freq_feats_dict, freq_feats_values

def extract_prosodic_phonation_features(y, sr):
    """提取：基频F0、Jitter、Shimmer、HNR、浊音段比例"""
    prosody_feats_dict = {}
    prosody_feats_values = []
    
    # 1. 基频F0
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'), sr=sr)
    f0_stats, f0_vals = calc_statistics(f0, "F0")
    prosody_feats_dict["基频(F0)"] = f0_stats
    prosody_feats_values.extend(f0_vals)
    
    # 2. 谐噪比（HNR）
    pw_f0, timeaxis = pw.harvest(y, sr)
    sp = pw.cheaptrick(y, pw_f0, timeaxis, sr)
    ap = pw.d4c(y, pw_f0, timeaxis, sr)
    y_harmonic = pw.synthesize(pw_f0, sp, np.zeros_like(ap), sr)
    harmonic_energy = np.sum(np.square(y_harmonic))
    total_energy = np.sum(np.square(y))
    noise_energy = total_energy - harmonic_energy + 1e-8
    hnr_value = 10 * np.log10(harmonic_energy / noise_energy) if harmonic_energy > 0 else 0.0
    hnr_stats, hnr_vals = calc_statistics(np.array([hnr_value]), "HNR")
    prosody_feats_dict["谐噪比(HNR)"] = hnr_stats
    prosody_feats_values.extend(hnr_vals)
    
    # 3. 基频微扰（Jitter）
    voiced_f0 = pw_f0[pw_f0 > 0]
    jitter_seq = np.abs(np.diff(voiced_f0)) / (np.mean(voiced_f0) + 1e-8) if len(voiced_f0) > 1 else np.array([0.0])
    jitter_stats, jitter_vals = calc_statistics(jitter_seq, "Jitter")
    prosody_feats_dict["基频微扰(Jitter)"] = jitter_stats
    prosody_feats_values.extend(jitter_vals)
    
    # 4. 振幅微扰（Shimmer）
    frame_energy = np.array([
        np.sum(np.square(y[i:i+FRAME_LENGTH] * WINDOW)) 
        for i in range(0, len(y)-FRAME_LENGTH, HOP_LENGTH)
    ])
    shimmer_seq = np.abs(np.diff(frame_energy)) / (np.mean(frame_energy) + 1e-8) if len(frame_energy) > 1 else np.array([0.0])
    shimmer_stats, shimmer_vals = calc_statistics(shimmer_seq, "Shimmer")
    prosody_feats_dict["振幅微扰(Shimmer)"] = shimmer_stats
    prosody_feats_values.extend(shimmer_vals)
    
    # 5. 浊音段比例
    voiced_frames = pw_f0 > 0
    voiced_ratio = np.sum(voiced_frames) / len(pw_f0) if len(pw_f0) > 0 else 0.0
    vr_stats, vr_vals = calc_statistics(np.array([voiced_ratio]), "VoicedRatio")
    prosody_feats_dict["浊音段比例(VoicedRatio)"] = vr_stats
    prosody_feats_values.extend(vr_vals)
    
    return prosody_feats_dict, prosody_feats_values

def extract_pause_temporal_features(y, sr):
    """提取：总停顿次数、平均停顿时长、总停顿时长占比"""
    pause_feats_dict = {}
    pause_feats_values = []
    
    # 停顿检测
    frame_energy = np.array([
        np.sum(np.square(y[i:i+FRAME_LENGTH] * WINDOW)) 
        for i in range(0, len(y)-FRAME_LENGTH, HOP_LENGTH)
    ])
    energy_threshold = np.mean(frame_energy) * 0.1 if len(frame_energy) > 0 else 0.0
    pause_frame_mask = frame_energy < energy_threshold
    
    # 统计停顿区间
    pause_intervals = []
    in_pause = False
    pause_start = 0
    for idx, is_pause in enumerate(pause_frame_mask):
        if is_pause and not in_pause:
            in_pause = True
            pause_start = idx
        elif not is_pause and in_pause:
            in_pause = False
            pause_intervals.append((pause_start, idx))
    if in_pause:
        pause_intervals.append((pause_start, len(pause_frame_mask)))
    
    total_duration = librosa.get_duration(y=y, sr=sr)
    total_pause_count = len(pause_intervals)
    
    # 1. 总停顿次数
    pc_stats, pc_vals = calc_statistics(np.array([total_pause_count]), "TotalPauseCount")
    pause_feats_dict["总停顿次数(TotalPauseCount)"] = pc_stats
    pause_feats_values.extend(pc_vals)
    
    # 2. 平均停顿时长
    avg_pause = np.mean([(end - start) * (HOP_LENGTH / sr) for (start, end) in pause_intervals]) if total_pause_count > 0 else 0.0
    ap_stats, ap_vals = calc_statistics(np.array([avg_pause]), "AvgPauseDuration")
    pause_feats_dict["平均停顿时长(AvgPauseDuration)"] = ap_stats
    pause_feats_values.extend(ap_vals)
    
    # 3. 总停顿时长占比
    pause_ratio = (np.sum([(end - start) * (HOP_LENGTH / sr) for (start, end) in pause_intervals]) / total_duration) if (total_pause_count > 0 and total_duration > 0) else 0.0
    pr_stats, pr_vals = calc_statistics(np.array([pause_ratio]), "PauseDurationRatio")
    pause_feats_dict["总停顿时长占比(PauseDurationRatio)"] = pr_stats
    pause_feats_values.extend(pr_vals)
    
    return pause_feats_dict, pause_feats_values

# -------------------------- 主函数：特征提取入口 --------------------------
def extract_suicide_risk_acoustic_features(audio_path, return_detail_stats=True):
    """自杀风险语音声学特征提取主函数"""
    # 1. 音频预处理
    y, sr = preprocess_audio(audio_path)
    
    # 2. 分模块提取特征
    time_dict, time_vals = extract_time_domain_features(y, sr)
    freq_dict, freq_vals = extract_frequency_cepstral_features(y, sr)
    prosody_dict, prosody_vals = extract_prosodic_phonation_features(y, sr)
    pause_dict, pause_vals = extract_pause_temporal_features(y, sr)
    
    # 3. 特征标准化
    all_raw_feats = np.array(time_vals + freq_vals + prosody_vals + pause_vals)
    mean_val = np.mean(all_raw_feats)
    std_val = np.std(all_raw_feats)
    final_feats = (all_raw_feats - mean_val) / std_val if std_val != 0 else all_raw_feats - mean_val
    
    # 4. 整理统计结果
    if return_detail_stats:
        detail_dict = {
            "time_domain": time_dict,
            "frequency_cepstral": freq_dict,
            "prosodic_phonation": prosody_dict,
            "pause_temporal": pause_dict,
            "global_summary": {
                "原始特征总维度": len(all_raw_feats),
                "标准化后均值": round(float(np.mean(final_feats)), 6),
                "标准化后标准差": round(float(np.std(final_feats)), 6),
                "各维度明细": FEATURE_CATEGORY_DESC["total"]["dimension_calc"],
                "维度说明": FEATURE_CATEGORY_DESC["total"]["note"]
            }
        }
        return final_feats, detail_dict
    return final_feats

# -------------------------- 测试与详细输出 --------------------------
if __name__ == "__main__":
    TEST_AUDIO_PATH = "/home/user512-003/songcw/code/scw-ser/data/wav/Ses01F_impro01/Ses01F_impro01_F000.wav"
    
    # 提取特征
    print("🔄 开始提取自杀风险语音声学特征...")
    final_features, detail_stats = extract_suicide_risk_acoustic_features(
        TEST_AUDIO_PATH, return_detail_stats=True
    )
    
    # 打印全局汇总
    print(f"\n{'='*100}")
    print("✅ 特征提取完成 - 全局汇总信息")
    print(f"{'='*100}")
    global_summary = detail_stats["global_summary"]
    print(f"📈 最终标准化特征维度：{global_summary['原始特征总维度']}")
    print(f"📊 标准化后全局均值：{global_summary['标准化后均值']}")
    print(f"📊 标准化后全局标准差：{global_summary['标准化后标准差']}")
    print(f"📏 维度构成明细：{global_summary['各维度明细']}")
    print(f"💡 维度说明：{global_summary['维度说明']}")
    
    # 打印各类特征详情（偏度/峰度已无nan）
    print_feature_detail_info(detail_stats["time_domain"], "time_domain")
    print_feature_detail_info(detail_stats["prosodic_phonation"], "prosodic_phonation")
    print_feature_detail_info(detail_stats["pause_temporal"], "pause_temporal")
    
    # 频域特征可选展示
    print(f"\n{'='*100}")
    print(f"📋 特征类别：{FEATURE_CATEGORY_DESC['frequency_cepstral']['name_cn']} ({FEATURE_CATEGORY_DESC['frequency_cepstral']['name_en']})")
    print(f"🔍 临床意义：{FEATURE_CATEGORY_DESC['frequency_cepstral']['clinical_significance']}")
    print(f"📏 维度构成：{FEATURE_CATEGORY_DESC['frequency_cepstral']['dimension_calc']}")
    print(f"💡 提示：频域特征维度较多，如需查看完整统计值，请取消下方注释")
    
    # 导出结果到JSON
    export_data = {
        "feature_category_description": FEATURE_CATEGORY_DESC,
        "global_summary": global_summary,
        "detailed_statistics": {
            "时域声学特征": detail_stats["time_domain"],
            "韵律与发声音质特征": detail_stats["prosodic_phonation"],
            "话语停顿与时序特征": detail_stats["pause_temporal"],
            "频域与倒谱特征": "详见代码内输出"
        }
    }
    with open("suicide_risk_audio_features_stats.json", "w", encoding="utf-8") as f:
        json.dump(export_data, f, ensure_ascii=False, indent=2)
    print(f"\n💾 详细统计结果已导出至：suicide_risk_audio_features_stats.json")
    print(f"{'='*100}")

🔄 开始提取自杀风险语音声学特征...

✅ 特征提取完成 - 全局汇总信息
📈 最终标准化特征维度：301
📊 标准化后全局均值：-0.0
📊 标准化后全局标准差：1.0
📏 维度构成明细：13(时域) + 240(频域) + 30(韵律) + 18(停顿) = 301维
💡 维度说明：维度修正说明：原373维为笔误，实际子特征统计为301维，核心特征体系不变

📋 特征类别：时域声学特征 (Time-domain Acoustic Features)
🔍 临床意义：反映语音的底层发声强度、节奏波动，自杀/抑郁高风险人群常表现为能量偏低、语速异常（过快/过慢/停顿多）
📏 维度构成：6(ZCR) + 6(短时能量) + 1(语速) = 13维
🧩 包含子特征：过零率(ZCR), 短时能量(ShortTermEnergy), 语速(SpeechRate)

📊 详细统计值：
--------------------------------------------------------------------------------

【过零率(ZCR)】
  均值    ：0.144346
  标准差   ：0.15778
  最大值   ：0.725
  最小值   ：0.01
  偏度    ：2.050004
  峰度    ：3.550289

【短时能量(ShortTermEnergy)】
  均值    ：0.00611
  标准差   ：0.017299
  最大值   ：0.093842
  最小值   ：2e-06
  偏度    ：3.309787
  峰度    ：10.838364

【语速(SpeechRate)】
  数值    ：100.228083

📋 特征类别：韵律与发声音质特征 (Prosodic & Phonation Quality Features)
🔍 临床意义：自杀风险最核心的声学标志物，高风险人群常表现为F0范围收窄、Jitter/Shimmer升高、HNR降低（声带振动不稳定）
📏 维度构成：6(F0) + 6(HNR) + 6(Jitter) + 6(Shimmer) + 6(VoicedRatio) = 30维
🧩 包含子特征：基频(F0), 谐噪比(HNR), 基频微扰(Jitter), 振幅微扰(Shim

/tmp/ipykernel_121137/3925392063.py:237: RuntimeWarning: invalid value encountered in log10
  hnr_value = 10 * np.log10(harmonic_energy / noise_energy) if harmonic_energy > 0 else 0.0
